In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/rag-from-scratch/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · The minimal RAG loop

The entire pattern, once:

```
embed the corpus  ─┐
embed the query  ──┤→ cosine similarity → top-k chunks → prompt → generate
```

We start with the crudest possible chunking — **one chunk per document** — so
nothing distracts from the loop. Notebook 02 fixes chunking.


In [ ]:
# --- setup: make `import ragkit` work from notebooks/ or solutions/ ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from ragkit.corpus import load_documents, load_corpus, load_qrels, tokenize
from ragkit.embed import get_embedder
from ragkit import llm

In [ ]:
emb = get_embedder()
chunks = load_corpus()                 # whole documents as chunks
matrix = emb.encode([c.text for c in chunks])   # (N, dim), rows normalised
print("index:", matrix.shape, "for", len(chunks), "chunks")

### Exercise 1 — cosine search

Because the rows of `matrix` are L2-normalised, cosine similarity is just the
dot product. Implement `search`: score every chunk against the query, return
the top-`k` as `(chunk, score)` pairs, highest first.


In [ ]:
def search(query, k=3):
    qv = emb.encode(query)                     # (dim,)
    scores = matrix @ qv                       # cosine, shape (N,)
    top = np.argsort(-scores)[:k]              # indices, best first
    return [(chunks[i], float(scores[i])) for i in top]

res = search("How many vacation days do I get each year?", k=3)
assert len(res) == 3
assert all(-1.01 <= s <= 1.01 for _, s in res)          # cosine range
assert res[0][1] >= res[1][1] >= res[2][1]              # sorted, descending
print("top-3:", [(c.doc_id, round(s, 3)) for c, s in res])

### Assemble a grounded prompt

Retrieval done. Now put the evidence in front of the model with a numbered
source list and an instruction to answer only from it (this is what makes the
answer *grounded* and *citable*). This part is plumbing, so it's written for
you — read it.


In [ ]:
def build_prompt(query, retrieved):
    blocks = []
    for i, (c, _) in enumerate(retrieved, 1):
        blocks.append(f"[{i}] (source: {c.doc_id})\n{c.text}")
    context = "\n\n".join(blocks)
    system = ("Answer the question using ONLY the sources below. "
              "Cite the source number in square brackets after each claim. "
              "If the sources don't contain the answer, say you don't know.")
    prompt = f"{system}\n\n=== SOURCES ===\n{context}\n\n=== QUESTION ===\n{query}"
    return prompt, [c.text for c, _ in retrieved]

p, ctxs = build_prompt("How many vacation days do I get each year?", res)
print(p[:600], "...")

### Exercise 2 — the end-to-end `answer()`

Tie it together: retrieve → build the prompt → generate. Use
`llm.complete(...)`, passing `contexts=` and `query=` so the offline fallback
has something to extract, and `embedder=` so it can rank sentences by meaning.


In [ ]:
def answer(query, k=3):
    retrieved = search(query, k)
    prompt, ctxs = build_prompt(query, retrieved)
    return llm.complete(prompt, contexts=ctxs, query=query, embedder=emb)

out = answer("How much can I get reimbursed for food abroad per day?")
assert isinstance(out, str) and len(out) > 0
print(out)

That's a working RAG system. Everything after this makes the **retrieve** step
better — because if the right text never makes it into `ctxs`, no amount of
prompting or model quality will save the answer.
